In [ ]:
# the Datacenter Computing team (@alexghergh)
!pip install datasets

from datasets import load_dataset
import torch
import torchvision
import pandas as pd
import matplotlib.pyplot as plt

torch.set_printoptions(precision=5, sci_mode=False)



mnist = load_dataset('ylecun/mnist')
mnist



idx = 148
img = mnist['train'][idx]['image']
label = mnist['train'][idx]['label']
img

# helper function to transform a 28x28 pixel image into a tensor
def image_to_tensor(img):
    # first we'll use a transform from torchvision
    tr = torchvision.transforms.PILToTensor()
    img = tr(img)

    # we transform to float and divide by 255 (so we work with values between 0 and 1, not 0 and 255)
    img = img.float() / 255
    return img

# we now have a tensor of size (1, 28, 28), of float values between 0 and 1
img_t = image_to_tensor(img)
img_t.shape, img_t

# let's visualize this tensor image more neatly
df = pd.DataFrame(img_t[0])
df.style.set_properties(**{'font-size':'6pt'}).background_gradient('Greys')

# see the docs: https://pytorch.org/docs/stable/generated/torch.reshape.html
img_t_resized = img_t.reshape(shape=(1,28*28))
img_t.shape, img_t_resized.shape

model = torch.nn.Sequential(
    torch.nn.Linear(28*28, 10),
)
model

sum(p.numel() for p in model.parameters())

loss_fn = torch.nn.CrossEntropyLoss()
loss_fn

optimizer = torch.optim.SGD(model.parameters(), lr=0.005)
optimizer

# we need to transform our data a bit before passing to the DataLoader, because
# a DataLoader only accepts tensors, but we have images; we'll use the same transformation
# from above, but this time on the whole dataset
def preprocess_data(example):
    for i in range(len(example['image'])):
        # convert image to tensor
        example['image'][i] = image_to_tensor(example['image'][i])
        # reshape into an array/vector, instead of a matrix
        example['image'][i] = example['image'][i].reshape(shape=(1, 28*28))
    return example

# preprocess the dataset
mnist = mnist.with_transform(preprocess_data)

# create the dataloader (we'll also shuffle the images, as it helps the model see images in random order)
dl = torch.utils.data.DataLoader(mnist['train'], batch_size=64, shuffle=True)
dl

epochs = 3
losses = []

for epoch in range(epochs):

    for batch in dl:
        # retrieve one batch of data from the dataloader
        inputs, targets = batch['image'].squeeze(dim=1), batch['label']

        # before anything else, make sure we have no gradients yet calculated/saved,
        # for any of the model parameters
        optimizer.zero_grad()

        # calculate the output of the model
        output = model(inputs)

        # calculate the loss (error) of the model with respect to the image labels (targets)
        loss = loss_fn(output, targets)

        # print and save this loss value
        print('loss: ', loss.item())
        losses.append(loss.item())

        # this is where the magic happens; PyTorch now calculates the gradients AUTOMATICALLY for us,
        # and the optimizer updates the parameters with these gradients
        loss.backward()
        optimizer.step()

plt.scatter(range(len(losses)), losses)

# set the model in evaluation mode; this tells PyTorch "do not calculate the gradients for me,
# as I do not need these now"
model.eval()

# let's also create a simple dataloader for the test set; although not necessary, it's easier
# to work with one (we'll evaluate one example at a time, and we don't need shuffling)
test_dl = torch.utils.data.DataLoader(mnist['test'], batch_size=1)

correct = 0
total = 0

with torch.no_grad():
    # iterate over the test set images
    for batch in test_dl:
        # get the input image and the label
        img, target = batch['image'].squeeze(dim=1), batch['label']

        # pass the image through the model
        output = model(img)

        # get the predicted class (the highest probability among the 10 classes of digits)
        predicted_class = torch.argmax(output, dim=1)

        # if correct, add 1
        total += 1
        correct += (predicted_class == target).sum().item()

accuracy = correct / total
print(f'accuracy: {accuracy * 100:.2f}%')

model = torch.nn.Sequential(
    torch.nn.Linear(28*28, 2 * 28*28),
    torch.nn.ReLU(),
    torch.nn.Linear(2 * 28*28, 4 * 28*28),
    torch.nn.ReLU(),
    torch.nn.Linear(4 * 28*28, 2 * 28*28),
    torch.nn.ReLU(),
    torch.nn.Linear(2 * 28*28, 28*28),
    torch.nn.ReLU(),
    torch.nn.Linear(28*28, 10),
)
model